Cell 1: Imports

In [6]:
import os
import json
import pandas as pd
from pathlib import Path
import cv2

Cell 2: Set paths

In [7]:
DATASET_ROOT = Path("D:/Judy Uni/archive/PVDN")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Dataset root exists:", DATASET_ROOT.exists())

Dataset root exists: True


In [8]:
# Build fast image lookup table

image_lookup = {}

possible_extensions = [".png", ".jpg", ".jpeg", ".bmp"]

all_images = []

for split in ["train", "val", "test"]:
    
    images_dir = (
        DATASET_ROOT /
        "night" /
        split /
        "images"
    )
    
    for ext in possible_extensions:
        all_images.extend(
            list(images_dir.rglob(f"*{ext}"))
        )

print("Total images found:", len(all_images))

for img_path in all_images:
    image_lookup[img_path.stem] = img_path

print("Lookup table created.")

Total images found: 33638
Lookup table created.


Cell 3: Helper functions

In [9]:
def find_image_by_annotation(annotation_path):
    """
    Fast lookup using prebuilt dictionary
    """
    
    image_stem = annotation_path.stem
    
    return image_lookup.get(image_stem, None)

In [10]:
def analyze_annotation(annotation_path):
    with open(annotation_path, "r") as f:
        data = json.load(f)
    
    vehicles = data.get("annotations", [])
    
    has_vehicle = len(vehicles) > 0
    has_visible_vehicle = False
    has_hidden_vehicle = False
    
    num_vehicles = len(vehicles)
    num_visible_vehicles = 0
    num_hidden_vehicles = 0
    
    num_direct_lights = 0
    num_reflections = 0
    num_oncoming_reflections = 0
    num_rear_reflections = 0
    
    has_oncoming_reflection = False
    has_reflection_only_signal = False
    
    vehicle_positions = []
    direct_light_positions = []
    reflection_positions = []
    
    for vehicle in vehicles:
        vehicle_direct = vehicle["direct"]
        vehicle_positions.append(vehicle["pos"])
        
        if vehicle_direct:
            has_visible_vehicle = True
            num_visible_vehicles += 1
        else:
            has_hidden_vehicle = True
            num_hidden_vehicles += 1
        
        for instance in vehicle.get("instances", []):
            instance_direct = instance["direct"]
            instance_rear = instance["rear"]
            
            if instance_direct:
                num_direct_lights += 1
                direct_light_positions.append(instance["pos"])
            else:
                num_reflections += 1
                reflection_positions.append(instance["pos"])
                
                if instance_rear:
                    num_rear_reflections += 1
                else:
                    num_oncoming_reflections += 1
                    has_oncoming_reflection = True
                    
                    if vehicle_direct == False:
                        has_reflection_only_signal = True
    reflection_only_instances = 0

    for vehicle in vehicles:
        
        vehicle_hidden = vehicle["direct"] == False
        
        for instance in vehicle.get("instances", []):
            
            if (
                vehicle_hidden
                and instance["direct"] == False
                and instance["rear"] == False
            ):
                reflection_only_instances += 1
    reflection_density = (
    num_oncoming_reflections /
    max(num_vehicles, 1)
)
    
    return {
        "image_id": data.get("image_id"),
        "has_vehicle": int(has_vehicle),
        "has_visible_vehicle": int(has_visible_vehicle),
        "has_hidden_vehicle": int(has_hidden_vehicle),
        "num_vehicles": num_vehicles,
        "num_visible_vehicles": num_visible_vehicles,
        "num_hidden_vehicles": num_hidden_vehicles,
        "num_direct_lights": num_direct_lights,
        "num_reflections": num_reflections,
        "num_oncoming_reflections": num_oncoming_reflections,
        "num_rear_reflections": num_rear_reflections,
        "has_oncoming_reflection": int(has_oncoming_reflection),
        "has_reflection_only_signal": int(has_reflection_only_signal),
        "vehicle_positions": json.dumps(vehicle_positions),
        "direct_light_positions": json.dumps(direct_light_positions),
        "reflection_positions": json.dumps(reflection_positions),
        "reflection_only_instances": reflection_only_instances,
        "reflection_density": reflection_density,
    }

Cell 4: Build index for one split

In [11]:
def build_split_index(dataset_root, split_name):
    """
    split_name should be:
    train, val, or test
    """
    
    split_path = dataset_root / "night" / split_name
    
    images_dir = split_path / "images"
    keypoints_dir = split_path / "labels" / "keypoints"
    
    annotation_files = sorted(list(keypoints_dir.glob("*.json")))
    
    rows = []
    
    print(f"Building index for: {split_name}")
    print("Annotation files:", len(annotation_files))
    
    missing_images = 0
    
    for annotation_path in annotation_files:
        image_path = find_image_by_annotation(annotation_path)
        
        if image_path is None:
            missing_images += 1
            continue
        
        info = analyze_annotation(annotation_path)
        
        row = {
            "split": split_name,
            "image_path": str(image_path),
            "annotation_path": str(annotation_path),
            "image_filename": image_path.name,
            "sequence_folder": image_path.parent.name,
        }
        
        row.update(info)
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    print("Rows created:", len(df))
    print("Missing images:", missing_images)
    
    return df

Cell 5: Build train, val, test indexes

In [12]:
train_df = build_split_index(DATASET_ROOT, "train")
val_df = build_split_index(DATASET_ROOT, "val")
test_df = build_split_index(DATASET_ROOT, "test")

Building index for: train
Annotation files: 25264
Rows created: 25264
Missing images: 0
Building index for: val
Annotation files: 4322
Rows created: 4322
Missing images: 0
Building index for: test
Annotation files: 4052
Rows created: 4052
Missing images: 0


Cell 6: Preview train dataframe

In [13]:
train_df.head()

,split,image_path,annotation_path,image_filename,sequence_folder,image_id,has_vehicle,has_visible_vehicle,has_hidden_vehicle,num_vehicles,...,num_reflections,num_oncoming_reflections,num_rear_reflections,has_oncoming_reflection,has_reflection_only_signal,vehicle_positions,direct_light_positions,reflection_positions,reflection_only_instances,reflection_density
0,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000001.png,S00000,1,0,0,0,0,...,0,0,0,0,0,[],[],[],0,0.0
1,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000003.png,S00000,3,0,0,0,0,...,0,0,0,0,0,[],[],[],0,0.0
2,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000005.png,S00000,5,0,0,0,0,...,0,0,0,0,0,[],[],[],0,0.0
3,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000007.png,S00000,7,0,0,0,0,...,0,0,0,0,0,[],[],[],0,0.0
4,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000009.png,S00000,9,0,0,0,0,...,0,0,0,0,0,[],[],[],0,0.0


In [16]:
train_df[train_df["has_vehicle"] == 1].head()

,split,image_path,annotation_path,image_filename,sequence_folder,image_id,has_vehicle,has_visible_vehicle,has_hidden_vehicle,num_vehicles,...,num_reflections,num_oncoming_reflections,num_rear_reflections,has_oncoming_reflection,has_reflection_only_signal,vehicle_positions,direct_light_positions,reflection_positions,reflection_only_instances,reflection_density
10,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000021.png,S00000,21,1,0,1,1,...,1,1,0,1,1,"[[613, 536]]",[],"[[613, 527]]",1,1.0
11,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000023.png,S00000,23,1,0,1,1,...,1,1,0,1,1,"[[613, 535]]",[],"[[613, 527]]",1,1.0
12,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000025.png,S00000,25,1,0,1,1,...,1,1,0,1,1,"[[614, 534]]",[],"[[614, 523]]",1,1.0
13,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000027.png,S00000,27,1,0,1,1,...,1,1,0,1,1,"[[615, 534]]",[],"[[614, 524]]",1,1.0
14,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000029.png,S00000,29,1,0,1,1,...,1,1,0,1,1,"[[616, 534]]",[],"[[616, 521]]",1,1.0


In [17]:
train_df[train_df["has_reflection_only_signal"] == 1].head()

,split,image_path,annotation_path,image_filename,sequence_folder,image_id,has_vehicle,has_visible_vehicle,has_hidden_vehicle,num_vehicles,...,num_reflections,num_oncoming_reflections,num_rear_reflections,has_oncoming_reflection,has_reflection_only_signal,vehicle_positions,direct_light_positions,reflection_positions,reflection_only_instances,reflection_density
10,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000021.png,S00000,21,1,0,1,1,...,1,1,0,1,1,"[[613, 536]]",[],"[[613, 527]]",1,1.0
11,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000023.png,S00000,23,1,0,1,1,...,1,1,0,1,1,"[[613, 535]]",[],"[[613, 527]]",1,1.0
12,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000025.png,S00000,25,1,0,1,1,...,1,1,0,1,1,"[[614, 534]]",[],"[[614, 523]]",1,1.0
13,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000027.png,S00000,27,1,0,1,1,...,1,1,0,1,1,"[[615, 534]]",[],"[[614, 524]]",1,1.0
14,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000029.png,S00000,29,1,0,1,1,...,1,1,0,1,1,"[[616, 534]]",[],"[[616, 521]]",1,1.0


In [18]:
print(train_df["has_vehicle"].value_counts())

has_vehicle
1    18272
0     6992
Name: count, dtype: int64


In [19]:
print(train_df["has_reflection_only_signal"].value_counts())

has_reflection_only_signal
0    18452
1     6812
Name: count, dtype: int64


Cell 7: Check statistics

In [14]:
def print_stats(df, name):
    print("=" * 50)
    print(name.upper())
    print("=" * 50)
    
    print("Total images:", len(df))
    print("Images with vehicles:", df["has_vehicle"].sum())
    print("Images with visible vehicles:", df["has_visible_vehicle"].sum())
    print("Images with hidden vehicles:", df["has_hidden_vehicle"].sum())
    print("Images with oncoming reflections:", df["has_oncoming_reflection"].sum())
    print("Images with reflection-only signal:", df["has_reflection_only_signal"].sum())
    
    print("\nTotal vehicles:", df["num_vehicles"].sum())
    print("Visible vehicles:", df["num_visible_vehicles"].sum())
    print("Hidden vehicles:", df["num_hidden_vehicles"].sum())
    
    print("\nDirect lights:", df["num_direct_lights"].sum())
    print("Reflections:", df["num_reflections"].sum())
    print("Oncoming reflections:", df["num_oncoming_reflections"].sum())
    print("Rear reflections:", df["num_rear_reflections"].sum())

In [23]:
full_df["label_reflection_only"] = (
    full_df["has_reflection_only_signal"]
)

train_df["label_reflection_only"] = (
    train_df["has_reflection_only_signal"]
)

val_df["label_reflection_only"] = (
    val_df["has_reflection_only_signal"]
)

test_df["label_reflection_only"] = (
    test_df["has_reflection_only_signal"]
)

Cell 8: Create full dataframe

In [24]:
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("\nFULL DATAFRAME INFO")
print("\n")

print(full_df.info())

print("Full dataset size:", len(full_df))
full_df.head()


FULL DATAFRAME INFO


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33638 entries, 0 to 33637
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   split                       33638 non-null  object 
 1   image_path                  33638 non-null  object 
 2   annotation_path             33638 non-null  object 
 3   image_filename              33638 non-null  object 
 4   sequence_folder             33638 non-null  object 
 5   image_id                    33638 non-null  int64  
 6   has_vehicle                 33638 non-null  int64  
 7   has_visible_vehicle         33638 non-null  int64  
 8   has_hidden_vehicle          33638 non-null  int64  
 9   num_vehicles                33638 non-null  int64  
 10  num_visible_vehicles        33638 non-null  int64  
 11  num_hidden_vehicles         33638 non-null  int64  
 12  num_direct_lights           33638 non-null  int64  
 13  num_refl

,split,image_path,annotation_path,image_filename,sequence_folder,image_id,has_vehicle,has_visible_vehicle,has_hidden_vehicle,num_vehicles,...,num_oncoming_reflections,num_rear_reflections,has_oncoming_reflection,has_reflection_only_signal,vehicle_positions,direct_light_positions,reflection_positions,reflection_only_instances,reflection_density,label_reflection_only
0,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000001.png,S00000,1,0,0,0,0,...,0,0,0,0,[],[],[],0,0.0,0
1,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000003.png,S00000,3,0,0,0,0,...,0,0,0,0,[],[],[],0,0.0,0
2,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000005.png,S00000,5,0,0,0,0,...,0,0,0,0,[],[],[],0,0.0,0
3,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000007.png,S00000,7,0,0,0,0,...,0,0,0,0,[],[],[],0,0.0,0
4,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000009.png,S00000,9,0,0,0,0,...,0,0,0,0,[],[],[],0,0.0,0


In [25]:
print("\nCLASS BALANCE")
print("=" * 50)

print(
    full_df["label_reflection_only"]
    .value_counts(normalize=True)
)


CLASS BALANCE
label_reflection_only
0    0.740977
1    0.259023
Name: proportion, dtype: float64


In [26]:
print(full_df.describe())

            image_id   has_vehicle  has_visible_vehicle  has_hidden_vehicle  \
count   33638.000000  33638.000000         33638.000000        33638.000000   
mean    60737.184821      0.706968             0.403888            0.379363   
std     44793.411137      0.455160             0.490683            0.485236   
min         1.000000      0.000000             0.000000            0.000000   
25%     20914.500000      0.000000             0.000000            0.000000   
50%     51481.000000      1.000000             0.000000            0.000000   
75%     87940.500000      1.000000             1.000000            1.000000   
max    143383.000000      1.000000             1.000000            1.000000   

       num_vehicles  num_visible_vehicles  num_hidden_vehicles  \
count  33638.000000          33638.000000         33638.000000   
mean       0.998841              0.483085             0.515756   
std        1.025683              0.674648             0.845730   
min        0.000000     

Cell 9: Create reflection-only subset

In [27]:
reflection_only_df = full_df[full_df["has_reflection_only_signal"] == 1].copy()

print("Reflection-only samples:", len(reflection_only_df))
reflection_only_df.head()

Reflection-only samples: 8713


,split,image_path,annotation_path,image_filename,sequence_folder,image_id,has_vehicle,has_visible_vehicle,has_hidden_vehicle,num_vehicles,...,num_oncoming_reflections,num_rear_reflections,has_oncoming_reflection,has_reflection_only_signal,vehicle_positions,direct_light_positions,reflection_positions,reflection_only_instances,reflection_density,label_reflection_only
10,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000021.png,S00000,21,1,0,1,1,...,1,0,1,1,"[[613, 536]]",[],"[[613, 527]]",1,1.0,1
11,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000023.png,S00000,23,1,0,1,1,...,1,0,1,1,"[[613, 535]]",[],"[[613, 527]]",1,1.0,1
12,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000025.png,S00000,25,1,0,1,1,...,1,0,1,1,"[[614, 534]]",[],"[[614, 523]]",1,1.0,1
13,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000027.png,S00000,27,1,0,1,1,...,1,0,1,1,"[[615, 534]]",[],"[[614, 524]]",1,1.0,1
14,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000029.png,S00000,29,1,0,1,1,...,1,0,1,1,"[[616, 534]]",[],"[[616, 521]]",1,1.0,1


Cell 10: Create visible vehicle subset

In [28]:
visible_df = full_df[full_df["has_visible_vehicle"] == 1].copy()

print("Visible vehicle samples:", len(visible_df))
visible_df.head()

Visible vehicle samples: 13586


,split,image_path,annotation_path,image_filename,sequence_folder,image_id,has_vehicle,has_visible_vehicle,has_hidden_vehicle,num_vehicles,...,num_oncoming_reflections,num_rear_reflections,has_oncoming_reflection,has_reflection_only_signal,vehicle_positions,direct_light_positions,reflection_positions,reflection_only_instances,reflection_density,label_reflection_only
92,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000185.png,S00000,185,1,1,0,1,...,2,0,1,0,"[[611, 558]]","[[602, 552], [620, 552]]","[[443, 550], [709, 551]]",0,2.0,0
93,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000187.png,S00000,187,1,1,0,1,...,2,0,1,0,"[[609, 559]]","[[601, 553], [619, 554]]","[[439, 549], [710, 552]]",0,2.0,0
94,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000189.png,S00000,189,1,1,0,1,...,2,0,1,0,"[[609, 561]]","[[600, 555], [618, 556]]","[[434, 552], [710, 555]]",0,2.0,0
95,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000191.png,S00000,191,1,1,0,1,...,2,0,1,0,"[[608, 561]]","[[599, 556], [618, 556]]","[[431, 553], [710, 556]]",0,2.0,0
96,train,D:\Judy Uni\archive\PVDN\night\train\images\S0...,D:\Judy Uni\archive\PVDN\night\train\labels\ke...,000193.png,S00000,193,1,1,0,1,...,2,0,1,0,"[[608, 562]]","[[599, 557], [618, 557]]","[[431, 553], [710, 556]]",0,2.0,0


Cell 11: Create binary classification label

In [29]:
full_df["label_approaching_vehicle"] = full_df["has_vehicle"]

train_df["label_approaching_vehicle"] = train_df["has_vehicle"]
val_df["label_approaching_vehicle"] = val_df["has_vehicle"]
test_df["label_approaching_vehicle"] = test_df["has_vehicle"]

full_df[["image_path", "label_approaching_vehicle"]].head()

,image_path,label_approaching_vehicle
0,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
1,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
2,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
3,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
4,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0


Cell 12: Create stronger reflection-focused label

In [30]:
#1 = hidden approaching vehicle indicated by reflection
#0 = other
full_df["label_reflection_only"] = full_df["has_reflection_only_signal"]

train_df["label_reflection_only"] = train_df["has_reflection_only_signal"]
val_df["label_reflection_only"] = val_df["has_reflection_only_signal"]
test_df["label_reflection_only"] = test_df["has_reflection_only_signal"]

full_df[["image_path", "label_reflection_only"]].head()

,image_path,label_reflection_only
0,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
1,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
2,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
3,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0
4,D:\Judy Uni\archive\PVDN\night\train\images\S0...,0


Cell 13: Save CSV files

In [31]:
train_df.to_csv(OUTPUT_DIR / "train_index.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val_index.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_index.csv", index=False)
full_df.to_csv(OUTPUT_DIR / "full_index.csv", index=False)
reflection_only_df.to_csv(OUTPUT_DIR / "reflection_only_index.csv", index=False)
visible_df.to_csv(OUTPUT_DIR / "visible_vehicle_index.csv", index=False)

print("Saved CSV files to:", OUTPUT_DIR.resolve())

Saved CSV files to: D:\Judy Uni\archive\outputs


Cell 14: Verify saved files

In [32]:
for file in OUTPUT_DIR.glob("*.csv"):
    print(file.name)

full_index.csv
reflection_only_index.csv
test_index.csv
train_index.csv
val_index.csv
visible_vehicle_index.csv
